# Capítulo 4 — El agente que olvida tu nombre

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Laura: *«Me llamo Laura»*. Un mensaje después: *«¿Cómo me llamo?»*. El agente no lo sabe.

| | |
|---|---|
| **🧠 Componente agéntico** | Memoria de trabajo y a corto plazo |
| **La necesidad** | Que el agente recuerde la conversación en curso |
| **Lo que construimos** | **AgentCore Memory** (corto plazo) con el *session manager* de Strands |
| **🏠 En local** | ✅ Funciona sin que lo notes: en el capítulo 2 el objeto `Agent` ya guardaba los mensajes en la RAM del proceso. Se rompe al desplegar, sin estado y con réplicas. |
| **☁️ Por qué AWS al salir a Internet** | Guardar el historial en un diccionario del proceso es literalmente RAM volátil: se pierde al reiniciar y, con dos réplicas detrás de un balanceador, el cliente cae en la otra y el agente «olvida». Memory es un servicio compartido por todas las réplicas. |
| **En el ordenador sería…** | La **RAM**: se puede escribir, pero es volátil. Hay que conservar lo que pasa mientras dura la sesión |
| **Requisitos del caso** | Base de RF-05 |
| **Prerrequisito · duración** | Capítulo 3 · 45 min |
| **⚠️ Dónde se rompe la analogía** | La RAM se lee sin coste; la ventana de contexto **se reprocesa entera en cada llamada y se paga cada vez**. De ahí el taxímetro. |

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(4)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(4)
# ponerse_al_dia(4)   # ← descomenta si llegas tarde o algo se rompió

## 1. Experimento de rotura: dos réplicas con la memoria en el proceso

En producción hay varias copias del agente detrás de un balanceador. Si la memoria vive dentro de cada copia, el segundo mensaje puede caer en otra réplica.

> ✋ **Predice antes de ejecutar.** La réplica A sabrá cómo te llamas. ¿Y la réplica B? Dilo en voz alta o apúntalo antes de lanzar la celda.

In [ ]:
from strands import Agent
from strands.models import BedrockModel

def replica():
    return Agent(model=BedrockModel(model_id=state_get("MODEL_ID"), region_name=region(), temperature=0.2),
                 system_prompt="Eres el asistente de TiendaNova.", callback_handler=None)

replica_a, replica_b = replica(), replica()
print("Turno 1 → réplica A:", replica_a("Hola, me llamo Laura."))
print("Turno 2 → réplica B:", replica_b("¿Cómo me llamo?"))

La réplica B no sabe tu nombre: la memoria vivía **dentro** de la réplica A.

> 🧭 **¿Por qué ha pasado?** Abre la teoría: [**La RAM del agente**](teoria/cap04-memoria-corto-plazo.html). Repite la conversación de Laura en tres sitios (en tu máquina, en Internet sin memoria y en Internet con Memory) y vuelve cuando sepas responder a esta pregunta: **¿dónde tiene que vivir la conversación para que cualquier réplica la recuerde?**

**En dos líneas:** la *memoria de trabajo* es lo que el modelo ve en esta llamada; la *memoria a corto plazo* es el historial que hay que guardar en algún sitio y reenviar en cada llamada. Al desplegar sin estado y con réplicas, ese sitio no puede ser la RAM de un proceso: tiene que ser un servicio compartido.

## 2. Crear la memoria con la CLI

Se crea ya con estrategias de largo plazo (las usará el capítulo 5). Así la CLI gestiona también los permisos del Runtime sobre la memoria.

In [ ]:
!cd "{PROJECT}" && agentcore add memory --name NovaMemory --strategies SEMANTIC,SUMMARIZATION,USER_PREFERENCE
!cd "{PROJECT}" && agentcore deploy --yes

In [ ]:
mid = buscar_memoria("NovaMemory")
if mid:
    state_set(MEMORY_ID=mid)
# Si no la encuentra: state_set(MEMORY_ID="<ID de agentcore status>")

## 3. Código

In [ ]:
fijar_dependencia(VERSIONES["bedrock-agentcore"])
write_settings()

In [ ]:
%%writefile NovaAssist/app/NovaAgent/memoria.py
"""Memoria de AgentCore para Strands: corto plazo (eventos) y largo plazo (recuperación por namespaces)."""
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager

from settings import LTM_NAMESPACES, MEMORY_ID, REGION


def retrieval_config() -> dict:
    """Vacío en el capítulo 4 (solo corto plazo); en el capítulo 5 se rellena desde LTM_NAMESPACES."""
    return {ns: RetrievalConfig(top_k=v[0], relevance_score=v[1]) for ns, v in LTM_NAMESPACES.items()}


def crear_session_manager(actor_id: str, session_id: str, retrieval: dict | None = None):
    kwargs = {"memory_id": MEMORY_ID, "session_id": session_id, "actor_id": actor_id}
    if retrieval:
        kwargs["retrieval_config"] = retrieval
    return AgentCoreMemorySessionManager(AgentCoreMemoryConfig(**kwargs), region_name=REGION)

In [ ]:
%%writefile NovaAssist/app/NovaAgent/main.py
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands import Agent
from strands.models import BedrockModel

from herramientas import HERRAMIENTAS
from memoria import crear_session_manager, retrieval_config
from prompts import SYSTEM_PROMPT
from settings import MODEL_ID, REGION

app = BedrockAgentCoreApp()


def metricas(result):
    """Herramientas usadas y consumo de tokens (para la verificación en los notebooks)."""
    try:
        m = result.metrics
        return sorted(m.tool_metrics.keys()), dict(m.accumulated_usage)
    except Exception:  # noqa: BLE001
        return [], {}


@app.entrypoint
def invoke(payload, context):
    actor_id = payload.get("cliente_id", "anonimo")  # ⚠️ provisional: en el capítulo 9 saldrá del token
    with crear_session_manager(actor_id, context.session_id, retrieval_config()) as sm:
        agent = Agent(
            model=BedrockModel(model_id=MODEL_ID, region_name=REGION, temperature=0.2),
            system_prompt=SYSTEM_PROMPT,
            tools=HERRAMIENTAS,
            session_manager=sm,
            callback_handler=None,
        )
        result = agent(payload.get("prompt", ""))
    herramientas, uso = metricas(result)
    return {"respuesta": str(result), "herramientas": herramientas, "tokens": uso}


if __name__ == "__main__":
    app.run()

## 4. Prueba local: misma sesión frente a sesión nueva

In [ ]:
sid = new_session_id()
probar_local("Me llamo Laura", session_id=sid)
probar_local("¿Cómo me llamo?", session_id=sid)          # misma sesión → lo recuerda
probar_local("¿Cómo me llamo?", session_id=new_session_id())   # sesión nueva → no lo sabe

## 5. Desplegar y repetir

In [ ]:
!cd "{PROJECT}" && agentcore deploy --yes

In [ ]:
sid = new_session_id()
invocar("Me llamo Laura", session_id=sid)
invocar("¿Cómo me llamo?", session_id=sid)

**Verificación adicional:** en la consola de AgentCore, abre **Memory → NovaMemory** y localiza los eventos de la sesión.

> `context.session_id` es el `runtimeSessionId`. Si tu versión del SDK lo expone con otro nombre, consulta la referencia de `RequestContext`. Si aparece `AccessDenied` sobre Memory en el Runtime, revisa los permisos del rol de ejecución.

## Para pensar

- ¿Qué ocurre si dos clientes comparten `actor_id`?

## 🔎 Lo que acabamos de descubrir

Laura vuelve **mañana**, en una sesión nueva, y el agente no recuerda que prefiere WhatsApp. Lo que guardamos dura lo que dura la sesión. Además, el `cliente_id` sigue llegando en el payload: cualquiera puede escribir el de otro.

In [ ]:
deuda("D4", "abierta", 4, "cliente_id en el payload")
cuaderno()
taximetro_cerrar(4)
guardar_partida(4)

---
➡️ **Siguiente capítulo:** `05_memoria_largo_plazo.ipynb` — *El agente que no te reconoce mañana*